In [ ]:
from data_processing.dot_env import config
from pathlib import Path
import re
import pandas as pd

In [ ]:
id_numbers=[423]

In [ ]:
maybe_input = config.get("INPUT_DATA_FOLDER")
INPUT_DATA_FOLDER = Path(maybe_input if maybe_input is not None else "fix_input")
OUTPUT_DATA_FOLDER = Path.home() / "Desktop"

In [ ]:
ids=[f"ID-{number}" for number in id_numbers]
input_paths=[INPUT_DATA_FOLDER / id / "raw_data" / "RAW" for id in ids]
output_paths=[OUTPUT_DATA_FOLDER / "Sergey Converted Raw Data" / id for id in ids]

In [ ]:
for output_path in output_paths:
    output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
END_NUMBER_PATTERN = r"^(.*_)(\d+)$"
SAMPLES_COL_NAME = "SAMPLES"
DELIMITER = ";"

for input_path, output_path in zip(input_paths, output_paths):
    # find first file, get headers
    source_files = [f for f in input_path.iterdir() if f.is_file and f.suffix.lower() == '.csv']
    first_file = [f for f in source_files if re.match(END_NUMBER_PATTERN, f.stem) is None]
    if len(first_file) == 0:
        print(f"Cannot find headers for files at {input_path}")
        continue
    first_file = first_file[0]
    with open(first_file, "r") as openfile:
        header_line = openfile.readline()
        data_line = openfile.readline()
    headers = header_line.strip().split(DELIMITER)
    data_sample = data_line.strip().split(DELIMITER)
    total_cols = len(data_sample)
    psd_cols = [col for col in headers if col != SAMPLES_COL_NAME]
    signal_cols = [str(n) for n in range(total_cols - len(psd_cols))]
    all_cols = psd_cols + signal_cols
    
    # load all csvs as parquet
    for file in source_files:
        has_header_line = re.match(END_NUMBER_PATTERN, file.stem) is None
        header = 0 if has_header_line else None
        df_raw = pd.read_csv(
            file,
            sep=DELIMITER,
            header=header,
            names=all_cols,
            dtype=str,
            on_bad_lines="skip"
        )
        print(file)
        df_raw.to_parquet(output_path / f"{file.stem}.parquet")